In [2]:
import pandas as pd

df = pd.read_csv("retail_sales_analysis_clean.csv")

print("Dataset shape:", df.shape)

Dataset shape: (62884, 35)


In [52]:
# Convert Order Date to datetime for time-based analysis
df["Order Date"] = pd.to_datetime(df["Order Date"])

In [54]:
df.head()

,Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code,Product Name,...,Continent,Birthday,Store Country,Store State,Square Meters,Open Date,Revenue USD,Cost USD,Gross Profit USD,Year-Month
0,366000,1,2016-01-01,NaN,265598,10,1304,1,CAD,Contoso Lens Adapter M450 White,...,North America,3/23/1971,Canada,Nunavut,1210.0,2015-04-04,68.00,31.27,36.73,2016-01
1,366001,1,2016-01-01,2016-01-13,1269051,0,1048,2,USD,A. Datum SLR Camera X136 Silver,...,North America,11/20/1995,Online,Online,NaN,2010-01-01,854.00,282.94,571.06,2016-01
2,366001,2,2016-01-01,2016-01-13,1269051,0,2007,1,USD,Fabrikam Microwave 1.5CuFt X1100 Black,...,North America,11/20/1995,Online,Online,NaN,2010-01-01,665.94,220.64,445.30,2016-01
3,366002,1,2016-01-01,2016-01-12,266019,0,1106,7,CAD,Contoso SLR Camera M146 Orange,...,North America,8/24/1962,Online,Online,NaN,2010-01-01,2254.00,1036.56,1217.44,2016-01
4,366002,2,2016-01-01,2016-01-12,266019,0,373,1,CAD,Adventure Works Laptop8.9 E0890 White,...,North America,8/24/1962,Online,Online,NaN,2010-01-01,326.00,166.20,159.80,2016-01


In [6]:
#First, identify the revenue column:- Revenue USD, Cost USD, Gross profit USD

In [8]:
df.columns.tolist()

['Order Number',
 'Line Item',
 'Order Date',
 'Delivery Date',
 'CustomerKey',
 'StoreKey',
 'ProductKey',
 'Quantity',
 'Currency Code',
 'Product Name',
 'Brand',
 'Color',
 'Unit Cost USD',
 'Unit Price USD',
 'SubcategoryKey',
 'Subcategory',
 'CategoryKey',
 'Category',
 'Gender',
 'Name',
 'City',
 'State Code',
 'Customer State',
 'Zip Code',
 'Customer Country',
 'Continent',
 'Birthday',
 'Store Country',
 'Store State',
 'Square Meters',
 'Open Date',
 'Revenue USD',
 'Cost USD',
 'Gross Profit USD',
 'Year-Month']

In [5]:
#Total revenue generated by the retailer. Here we create a reusable python function so AI assistant can call it whenever someone asks about total revenue. 

In [50]:
def get_total_revenue(data):
    total_revenue = data["Revenue USD"].sum()
    return round(total_revenue, 2)
    
get_total_revenue(df)

55755479.59

In [12]:
#Gross Profit function: 

In [14]:
def get_total_gross_profit(data):
    total_profit = data["Gross Profit USD"].sum()
    return round(total_profit, 2)
    
get_total_gross_profit(df)

32662688.38

In [9]:
#gross_profit_margin

In [16]:
def get_gross_profit_margin(data):
    revenue = get_total_revenue(data)
    gross_profit = get_total_gross_profit(data)

    if revenue == 0:
        return 0

    margin = (gross_profit / revenue) * 100
    return round(margin, 2)
    
get_gross_profit_margin(df)

58.58

In [18]:
#total_orders

In [20]:
def get_total_orders(data):
    total_orders = data["Order Number"].nunique()
    return total_orders
    
get_total_orders(df)

26326

In [13]:
#average_order_value

In [56]:
def get_average_order_value(data):
    revenue = get_total_revenue(data)
    orders = get_total_orders(data)
    
    if orders == 0:
        return 0
        
    aov = revenue / orders
    return round(aov, 2)
    
get_average_order_value(df)

2117.89

In [15]:
#Revenue function:Before filtering by year, we need to verify its data type

In [26]:
# Before AI assistant can filter revenue by year

In [64]:
# Convert Order Date from text to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])

In [62]:
df["Order Date"].dt.year

0        2016
1        2016
2        2016
3        2016
4        2016
         ... 
62879    2021
62880    2021
62881    2021
62882    2021
62883    2021
Name: Order Date, Length: 62884, dtype: int32

In [20]:
#Dynamic revenue fun:total revenue in a specific year without changing the underlying code.

In [66]:
def get_revenue_by_year(data, year):
    filtered_data = data[data["Order Date"].dt.year == year]

    revenue = filtered_data["Revenue USD"].sum()

    return round(revenue, 2)

In [68]:
# Test the function for 2019
get_revenue_by_year(df, 2019)

18264382.48

In [70]:
get_revenue_by_year(df, 2020)

9294632.14

In [24]:
#Filter revenue by Year AND Product Category

In [74]:
def get_revenue_by_year_category(data, year, category):
    filtered_data = data[
        (data["Order Date"].dt.year == year) &
        (data["Category"] == category)
    ]

    revenue = filtered_data["Revenue USD"].sum()

    return round(revenue, 2)

# Test: Computers revenue in 2019
get_revenue_by_year_category(df, 2019, "Computers")

6958430.89

In [26]:
#Sales Channel filtering

In [76]:
# Combine year, product category, and sales channel filters.
# This allows the AI assistant to answer specific business questions
# without requiring a separate function for every filter combination.

In [82]:
def get_revenue_by_filters(data, year, category, channel):
    filtered_data = data[
        (data["Order Date"].dt.year == year) &
        (data["Category"] == category)
    ]

    if channel == "Online":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] == 0
        ]

    elif channel == "Physical Store":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] != 0
        ]

    else:
        raise ValueError("Channel must be 'Online' or 'Physical Store'.")

    revenue = filtered_data["Revenue USD"].sum()

    return round(revenue, 2)

In [86]:
# Example: Calculate Online Computers revenue in 2019
get_revenue_by_filters(df, 2019, "Computers", "Online")

1500897.15

In [90]:
# Validate Online vs. Physical Store Revenue — Computers, 2019

online = get_revenue_by_filters(
    df, 2019, "Computers", "Online"
)

physical = get_revenue_by_filters(
    df, 2019, "Computers", "Physical Store"
)

total = get_revenue_by_year_category(
    df, 2019, "Computers"
)

print("Online Revenue:", online)
print("Physical Store Revenue:", physical)
print("Combined Revenue:", round(online + physical, 2))
print("Total Computers Revenue:", total)
print("Validation:", round(online + physical, 2) == total)

Online Revenue: 1500897.15
Physical Store Revenue: 5457533.74
Combined Revenue: 6958430.89
Total Computers Revenue: 6958430.89
Validation: True


In [96]:
#Make the filters optional

def calculate_revenue(data, year=None, category=None, channel=None):
    filtered_data = data.copy()

    if year is not None:
        filtered_data = filtered_data[
            filtered_data["Order Date"].dt.year == year
        ]

    if category is not None:
        filtered_data = filtered_data[
            filtered_data["Category"] == category
        ]

    if channel == "Online":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] == 0
        ]

    elif channel == "Physical Store":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] != 0
        ]
    elif channel is not None:
        raise ValueError("Channel must be 'Online' or 'Physical Store'.")

    return round(filtered_data["Revenue USD"].sum(), 2)


print("2019 Revenue:", calculate_revenue(df, year=2019))

#testing with different questions

print("Computers Revenue:", calculate_revenue(df, category="Computers"))

print("Online Revenue 2020:", calculate_revenue(df, year=2020, channel="Online"))

print("Online Computers 2019:", calculate_revenue(
    df, year=2019, category="Computers", channel="Online"
))

2019 Revenue: 18264382.48
Computers Revenue: 19301595.46
Online Revenue 2020: 2069950.9
Online Computers 2019: 1500897.15


In [102]:
# Rule-based question interpretation before LLM integration

In [98]:
# Extract a year from a natural-language question

import re

def extract_year(question):
    match = re.search(r"\b(20\d{2})\b", question)

    if match:
        return int(match.group(1))

    return None

In [100]:
# Examples: Extract years from business questions

print(extract_year("What was revenue in 2019?"))
print(extract_year("Show me sales for 2020"))
print(extract_year("What is total revenue?"))

2019
2020
None


In [106]:
# Answer a revenue question using the extracted year

def answer_revenue_question(question, data):
    year = extract_year(question)

    revenue = calculate_revenue(data, year=year)

    return f"Revenue: ${revenue:,.2f}"

In [108]:
answer_revenue_question("What was revenue in 2019?", df)

'Revenue: $18,264,382.48'

In [114]:
# Extract product category from a natural-language question

def extract_category(question, data):
    categories = data["Category"].dropna().unique()

    for category in categories:
        if category.lower() in question.lower():
            return category

    return None

In [116]:
# Examples: Extract category from business questions

print(extract_category("What was Computers revenue in 2019?", df))
print(extract_category("Show me revenue for Cell phones", df))
print(extract_category("What was total revenue?", df))

Computers
Cell phones
None


In [118]:
#Connect category and year to revenue

def answer_revenue_question(question, data):
    year = extract_year(question)
    category = extract_category(question, data)

    revenue = calculate_revenue(
        data,
        year=year,
        category=category
    )

    return f"Revenue: ${revenue:,.2f}"

# Check
answer_revenue_question(
    "What was Computers revenue in 2019?", df
)

'Revenue: $6,958,430.89'

In [120]:
#What was Online Computers revenue in 2019?

def extract_channel(question):
    question = question.lower()

    if "online" in question:
        return "Online"

    elif "physical" in question or "in-store" in question:
        return "Physical Store"

    return None


print(extract_channel("Show online revenue in 2019"))
print(extract_channel("What were physical store sales?"))
print(extract_channel("What was total revenue?"))

Online
Physical Store
None


In [124]:
# Extract year, category, and sales channel from a business question
# and calculate revenue using the identified filters.

def answer_revenue_question(question, data):
    year = extract_year(question)
    category = extract_category(question, data)
    channel = extract_channel(question)

    revenue = calculate_revenue(
        data,
        year=year,
        category=category,
        channel=channel
    )

    return f"Revenue: ${revenue:,.2f}"


# Test
answer_revenue_question(
    "What was Online Computers revenue in 2019?", df
)

'Revenue: $1,500,897.15'

In [98]:
# Validate revenue answers across different business questions

questions = [
    "What was total revenue in 2019?",
    "What was Computers revenue in 2019?",
    "What was Online revenue in 2020?",
    "What was Physical Store Computers revenue in 2019?",
    "What was total revenue?"
]

for question in questions:
    print("Question:", question)
    print("Answer:", answer_revenue_question(question, df))
    print()

Question: What was total revenue in 2019?
Answer: Revenue: $18,264,382.48

Question: What was Computers revenue in 2019?
Answer: Revenue: $6,958,430.89

Question: What was Online revenue in 2020?
Answer: Revenue: $2,069,950.90

Question: What was Physical Store Computers revenue in 2019?
Answer: Revenue: $5,457,533.74

Question: What was total revenue?
Answer: Revenue: $55,755,479.59



In [128]:
import ollama

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": "What is revenue in business? Answer in one sentence."
        }
    ]
)

print(response["message"]["content"])

In business, revenue refers to the total amount of money earned by a company from its normal business operations, typically calculated by multiplying the total sales or turnover by the price at which goods or services are sold.


In [ ]:
#Test Ollama's understanding of our business questions

In [106]:
question = "How much did we make from online computer sales in 2019?"

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": f"""
            Identify the metric, year, category, and sales channel
            from this retail business question.

            Question: {question}

            Return only these four fields.
            """
        }
    ]
)

print(response["message"]["content"])

Here are the requested fields:

- Metric: Sales
- Year: 2019
- Category: Online computer sales
- Sales Channel: Online


In [ ]:
#Make Ollama return structured data

In [148]:
# Extract structured business information using Ollama
import json

question = "How much did we make from online computer sales in 2019?"

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": f"""
            Extract information from this retail business question.

            Question: {question}

            Return JSON with exactly these fields:
            metric, year, category, channel.

            Rules:
            - metric must be one of: revenue, cost, gross_profit
            - year must be a four-digit number or null
            - category must match one of: Computers, Home Appliances,
              Cameras and camcorders, Cell phones, TV and Video,
              Audio, Games and Toys, Music, Movies and Audio Books,
              or null
            - channel must be Online, Physical Store, or null
            - Interpret sales as revenue.
            - Do not invent missing filters.
            - Return JSON only.
            """
        }
    ],
    format="json"
)

result = json.loads(response["message"]["content"])

# Convert the extracted year to an integer
if result["year"] is not None:
    result["year"] = int(result["year"])

print(result)

{'metric': 'revenue', 'year': 2019, 'category': 'Computers', 'channel': 'Online'}


In [150]:
print("Dataset loaded:", df.shape)

Dataset loaded: (62884, 35)


In [152]:
print("Revenue function available:", callable(calculate_revenue))

Revenue function available: True


In [154]:
#Connecting Ollama to our revenue calculation

year = int(result["year"])
category = result["category"]
channel = result["channel"]

revenue = calculate_revenue(
    df,
    year=year,
    category=category,
    channel=channel
)

print(f"Question: {question}")
print(f"Revenue: ${revenue:,.2f}")

Question: How much did we make from online computer sales in 2019?
Revenue: $1,500,897.15


In [156]:
question = "What was Computers revenue in 2019?"

revenue = calculate_revenue(
    df,
    year=2019,
    category="Computers"
)

print(f"Question: {question}")
print(f"Revenue: ${revenue:,.2f}")

Question: What was Computers revenue in 2019?
Revenue: $6,958,430.89


In [166]:
# Extract business information from a natural-language question using Ollama

import ollama
import json

question = "What was Computers revenue in 2019?"

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": f"""
            Extract metric, year, category, and channel
            from this retail business question.

            Question: {question}

            Rules:
            - metric must be one of: revenue, cost, gross_profit
            - Interpret sales as revenue.
            - year must be a four-digit number or null.
            - category must match one of:
              Computers, Home Appliances,
              Cameras and camcorders, Cell phones,
              TV and Video, Audio, Games and Toys,
              Music, Movies and Audio Books
            - channel must be Online, Physical Store, or null.
            - Use null for filters not mentioned.
            - Do not invent missing information.
            - Return JSON only.
            """
        }
    ],
    format="json"
)

# Convert Ollama's JSON response into a Python dictionary
result = json.loads(response["message"]["content"])

# Convert string "null" values to Python None
for key in ["metric", "year", "category", "channel"]:
    if isinstance(result.get(key), str) and result[key].lower() == "null":
        result[key] = None

# Convert year to integer if present
if result["year"] is not None:
    result["year"] = int(result["year"])

print(result)

{'metric': 'revenue', 'year': 2019, 'category': 'Computers', 'channel': None}


In [168]:
#Connect AI interpretation to revenue calculation

revenue = calculate_revenue(
    df,
    year=result["year"],
    category=result["category"],
    channel=result["channel"]
)

print(f"Question: {question}")
print(f"Revenue: ${revenue:,.2f}")

Question: What was Computers revenue in 2019?
Revenue: $6,958,430.89


In [170]:
# Build a reusable AI Analytics Assistant function.
# Use Ollama to extract business filters from natural-language questions
# and pass those filters to Python for accurate calculations.

In [216]:
# Reusable AI assistant for retail revenue questions
def ask_retail_ai(question):
    valid_categories = df["Category"].dropna().unique().tolist()
    valid_channels = ["Online", "Physical Store"]

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": f"""
                Extract metric, year, category, and channel
                from this retail business question.

                Question: {question}

                Rules:
                - metric must be revenue, cost, or gross_profit.
                - Interpret sales as revenue.
                - category must match one of: {valid_categories}
                - channel must be Online, Physical Store, or null.
                - Return Online only if the question explicitly mentions online.
                - Return Physical Store only if the question explicitly mentions physical stores or in-store sales.
                - If no sales channel is mentioned, return null.
                - Never assume a sales channel.
                - year must be a four-digit year or null.
                - Use null for missing filters.
                - Do not invent missing information.
                - Return JSON only.
                """
            }
        ],
        format="json"
    )

    result = json.loads(response["message"]["content"])

    # Normalize missing values
    for key in ["metric", "year", "category", "channel"]:
        if isinstance(result.get(key), str):
            if result[key].strip().lower() == "null":
                result[key] = None

    # Validate the requested metric
    if result.get("metric") != "revenue":
        return "Currently, I can answer revenue questions only."

    # Validate the year
    year = result.get("year")

    if year is not None:
        try:
            year = int(year)
        except (ValueError, TypeError):
            return "Sorry, I could not identify a valid year."

    # Validate category and channel
    category = result.get("category")

# Determine sales channel directly from the user's question
# to prevent Ollama from inventing a channel filter.

    channel = extract_channel(question)

    if category is not None and category not in valid_categories:
        return "Sorry, I could not identify a valid product category."

    if channel is not None and channel not in valid_channels:
        return "Sorry, I could not identify a valid sales channel."

    # Calculate revenue using validated filters
    revenue = calculate_revenue(
        df,
        year=year,
        category=category,
        channel=channel
    )

    return f"Revenue: ${revenue:,.2f}"

In [218]:
# Test the complete AI Assistant with a single business question
ask_retail_ai("How much revenue did online computer sales generate in 2019?")

'Revenue: $1,500,897.15'

In [220]:
#Let's check whether the same function works when the question doesn't mention Online or Physical Store.

In [222]:
ask_retail_ai("What was the total revenue from Computers in 2019?")

'Revenue: $6,958,430.89'

In [224]:
ask_retail_ai("What was the Physical Store Computers revenue in 2019?")

'Revenue: $5,457,533.74'

In [226]:
ask_retail_ai("What was the gross profit in 2019?")

'Currently, I can answer revenue questions only.'

In [ ]:
#install Streamlit-to turn our working Python AI Assistant into a simple web application.


# Retail Sales AI Assistant — Final Development and Validation

## Objective

Build a simple AI-powered Retail Sales Assistant that understands natural-language business questions and returns accurate revenue calculations from the retail sales dataset.

## Tools Used

- **Python and Pandas:** Data filtering, validation, and revenue calculations.
- **Ollama (Llama 3.2):** Interpretation of natural-language business questions.
- **JSON:** Conversion of AI responses into structured information.
- **Streamlit:** Interactive web interface for asking questions and displaying answers.

## How the AI Assistant Works

1. The user enters a business question through the Streamlit interface.
2. Ollama interprets the question and extracts the metric, year, and product category.
3. Python identifies the sales channel directly from the question to prevent incorrect AI-generated filters.
4. Python validates the extracted information.
5. Pandas filters the retail dataset and calculates revenue.
6. Streamlit displays the calculated answer.

## Validated Results

The AI Assistant was tested using different business questions to verify revenue calculations and sales-channel filtering.

| Business Question | Revenue (USD) |

| Total Computers revenue in 2019 | $6,958,430.89 |
| Online Computers revenue in 2019 | $1,500,897.15 |
| Physical Store Computers revenue in 2019 | $5,457,533.74 |

The calculated results were verified against previously validated retail sales analysis results.

## Key Learnings

- Ollama interprets natural-language business questions and returns structured JSON.
- Python performs calculations using actual retail sales data rather than relying on AI-generated financial figures.
- Missing filters are handled using Python's `None` value.
- Python-based channel extraction prevents Ollama from inventing sales-channel filters.
- Category and channel validation helps prevent incorrect calculations.
- Separating AI interpretation from business calculations improves transparency, accuracy, and reliability.
- Streamlit provides a simple interface for users to ask business questions without writing Python code.


## Project Status

**Basic Retail Sales AI Assistant — Completed and Validated**

Successfully developed and tested the Retail Sales AI Assistant using Python, Pandas, Ollama (Llama 3.2), and Streamlit.

All six revenue test cases returned the expected results.

This is a working local prototype, not a production-ready application. Its current scope is revenue questions.  

## Limitations

- The assistant currently supports revenue questions only.
- AI interpretation may not always be accurate for unfamiliar question wording.
- The application runs locally and requires Ollama to be installed and running.
- The current implementation is a prototype designed for business analytics demonstrations rather than production deployment.

## Conclusion

This project demonstrates how a local large language model can interpret natural-language business questions while Python and Pandas perform reliable, data-driven calculations.

By separating AI interpretation from business logic, the Retail Sales AI Assistant provides a practical example of combining traditional data analytics with generative AI.

The Streamlit interface makes retail revenue analysis more accessible to users without requiring SQL or Python knowledge.

## Next Phase

Organize and finalize the complete Retail Sales Analytics portfolio project, including:

- PostgreSQL SQL analysis and data validation
- Python and Pandas analysis notebooks
- Tableau dashboard
- Retail Sales AI Assistant and Streamlit application
- GitHub repository documentation and README
- LinkedIn portfolio case study

**Final Goal:** Present an end-to-end Retail Sales Analytics + AI project demonstrating SQL, Python, Tableau, business intelligence, and practical AI integration.
